WiFi 802.11 Reinforcement Learning Testbed
A comprehensive virtual environment for learning adaptive channel allocation
and radio control policies in dense WLAN deployments.

Features:
- Multi-agent RL environment with CTDE support
- Realistic WiFi PHY/MAC modeling
- GNN-based policy architecture with proper edge features
- MAPPO training pipeline with standard value clipping
- Comprehensive evaluation metrics with CSV logging
- Model checkpointing and result plotting

In [ ]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool
from torch_geometric.data import Data, Batch
import gymnasium as gym
from gymnasium import spaces
import random
import math
import csv
import os
from datetime import datetime
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
from enum import Enum
import matplotlib.pyplot as plt
from collections import deque

# CONSTANTS & CONFIGURATION

In [ ]:
class Config:
    # Network parameters
    CHANNELS_24GHZ = [1, 6, 11]  # Non-overlapping 2.4GHz channels
    CHANNELS_5GHZ = [36, 40, 44, 48, 149, 153, 157, 161]
    ALL_CHANNELS = CHANNELS_24GHZ + CHANNELS_5GHZ
    CHANNEL_WIDTHS = [20, 40, 80]  # MHz
    TX_POWER_LEVELS = [10, 15, 20, 25, 30]  # dBm

    # Physical layer parameters
    THERMAL_NOISE = -104  # dBm
    CCA_THRESHOLD = -82   # dBm
    PATH_LOSS_EXPONENT = 2.5
    SHADOWING_STD = 8.0   # dB

    # Traffic models
    TRAFFIC_TYPES = ['voip', 'video', 'web', 'bulk']
    PACKET_SIZES = {'voip': 160, 'video': 1400, 'web': 800, 'bulk': 1500}
    ARRIVAL_RATES = {'voip': 100, 'video': 50, 'web': 20, 'bulk': 10}

    # RL parameters
    STATE_HISTORY_LEN = 10
    MAX_EPISODE_STEPS = 1000
    REWARD_WEIGHTS = {
        'throughput': 0.5,
        'latency': 0.2,
        'collision': 0.15,
        'switching': 0.1,
        'fairness': 0.05
    }

# DATA STRUCTURES

In [ ]:
@dataclass
class Station:
    id: int
    position: Tuple[float, float]
    ap_id: int
    traffic_type: str
    tx_power: float = 20.0
    data_rate: float = 54.0  # Mbps
    queue_length: int = 0
    last_transmission: float = 0.0
    backoff_slots: int = 0
    collision_count: int = 0
    packets_sent: int = 0
    packets_received: int = 0
    total_delay: float = 0.0

In [ ]:
@dataclass
class AccessPoint:
    id: int
    position: Tuple[float, float]
    channel: int = 6
    channel_width: int = 20
    tx_power: float = 25.0
    cca_threshold: float = -82.0
    stations: List[int] = field(default_factory=list)
    queue_length: int = 0
    busy_time: float = 0.0
    collision_count: int = 0
    throughput: float = 0.0
    last_channel_switch: float = 0.0

In [ ]:
@dataclass
class InterferenceSource:
    position: Tuple[float, float]
    power: float
    frequency: float
    bandwidth: float
    duty_cycle: float = 1.0

In [ ]:
class TrafficType(Enum):
    VOIP = "voip"
    VIDEO = "video"
    WEB = "web"
    BULK = "bulk"

# CSV LOGGER

In [ ]:
class CSVLogger:
    """CSV logging system replacing standard logging"""

    def __init__(self, log_dir: str = "/content/drive/MyDrive/WiFi_Generalization/wifi_rl_logs"):
        self.log_dir = log_dir
        os.makedirs(log_dir, exist_ok=True)

        # Create log files
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.training_log = os.path.join(log_dir, f"training_{timestamp}.csv")
        self.evaluation_log = os.path.join(log_dir, f"evaluation_{timestamp}.csv")
        self.episode_log = os.path.join(log_dir, f"episodes_{timestamp}.csv")

        # Initialize CSV files with headers
        self._init_csv_files()

    def _init_csv_files(self):
        # Training log
        with open(self.training_log, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['timestamp', 'update', 'policy_loss', 'value_loss', 'entropy_loss', 'total_loss'])

        # Evaluation log
        with open(self.evaluation_log, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['timestamp', 'agent', 'episode', 'total_reward', 'avg_throughput',
                           'avg_latency', 'avg_collision_rate', 'avg_fairness', 'channel_switches'])

        # Episode log
        with open(self.episode_log, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['timestamp', 'episode', 'steps', 'total_reward', 'avg_throughput',
                           'avg_latency', 'avg_collision_rate', 'avg_fairness'])

    def log_training(self, update: int, losses: Dict[str, float]):
        with open(self.training_log, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([
                datetime.now().isoformat(),
                update,
                losses.get('policy_loss', 0),
                losses.get('value_loss', 0),
                losses.get('entropy_loss', 0),
                losses.get('total_loss', 0)
            ])

    def log_evaluation(self, agent: str, episode: int, metrics: Dict[str, float]):
        with open(self.evaluation_log, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([
                datetime.now().isoformat(),
                agent,
                episode,
                metrics.get('total_reward', 0),
                metrics.get('avg_throughput', 0),
                metrics.get('avg_latency', 0),
                metrics.get('avg_collision_rate', 0),
                metrics.get('avg_fairness', 0),
                metrics.get('channel_switches', 0)
            ])

    def log_episode(self, episode: int, steps: int, metrics: Dict[str, float]):
        with open(self.episode_log, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([
                datetime.now().isoformat(),
                episode,
                steps,
                metrics.get('total_reward', 0),
                metrics.get('avg_throughput', 0),
                metrics.get('avg_latency', 0),
                metrics.get('avg_collision_rate', 0),
                metrics.get('avg_fairness', 0)
            ])

# PHYSICAL LAYER MODELS

In [ ]:
class RadioPropagationModel:
    """Models radio propagation with path loss, shadowing, and fading"""

    def __init__(self):
        self.shadowing_cache = {}

    def path_loss_db(self, distance_m: float, frequency_ghz: float = 2.4) -> float:
        """Calculate free space path loss"""
        if distance_m < 1.0:
            distance_m = 1.0
        return 20 * math.log10(distance_m) + 20 * math.log10(frequency_ghz) + 92.45

    def shadowing_db(self, tx_pos: Tuple[float, float],
                    rx_pos: Tuple[float, float]) -> float:
        """Correlated log-normal shadowing"""
        key = (tx_pos, rx_pos)
        if key not in self.shadowing_cache:
            self.shadowing_cache[key] = np.random.normal(0, Config.SHADOWING_STD)
        return self.shadowing_cache[key]

    def received_power(self, tx_power_dbm: float, tx_pos: Tuple[float, float],
                      rx_pos: Tuple[float, float]) -> float:
        """Calculate received power including all effects"""
        distance = math.sqrt((tx_pos[0] - rx_pos[0])**2 + (tx_pos[1] - rx_pos[1])**2)
        path_loss = self.path_loss_db(distance)
        shadowing = self.shadowing_db(tx_pos, rx_pos)
        return tx_power_dbm - path_loss + shadowing

In [ ]:
class InterferenceModel:
    """Models co-channel and adjacent channel interference"""

    def __init__(self):
        self.prop_model = RadioPropagationModel()

    def channel_separation_factor(self, ch1: int, ch2: int) -> float:
        """Adjacent channel rejection factor"""
        separation = abs(ch1 - ch2)
        if separation == 0:
            return 1.0  # Co-channel
        elif separation <= 5:
            return 0.1  # Adjacent channel
        else:
            return 0.01  # Well separated

    def calculate_sinr(self, signal_power: float, interferer_powers: List[float],
                      noise_power: float = -104) -> float:
        """Calculate Signal-to-Interference-plus-Noise Ratio"""
        interference = sum(interferer_powers) if interferer_powers else 0
        noise_linear = 10**(noise_power / 10)
        interference_linear = 10**(interference / 10) if interference > 0 else 0
        signal_linear = 10**(signal_power / 10)

        sinr_linear = signal_linear / (interference_linear + noise_linear)
        return 10 * math.log10(sinr_linear) if sinr_linear > 0 else -100

# WLAN ENVIRONMENT

In [ ]:
class WiFiEnvironment(gym.Env):
    """Multi-agent WiFi environment supporting CTDE training with fixed metrics"""

    def __init__(self, scenario_config: Dict):
        super().__init__()

        self.config = scenario_config
        self.num_aps = scenario_config.get('num_aps', 4)
        self.num_stations_per_ap = scenario_config.get('stations_per_ap', 5)
        self.area_size = scenario_config.get('area_size', (100, 100))
        self.max_steps = scenario_config.get('max_steps', Config.MAX_EPISODE_STEPS)

        # Time step duration (seconds)
        self.time_step_duration = 0.001  # 1ms per step

        # Initialize models
        self.prop_model = RadioPropagationModel()
        self.interference_model = InterferenceModel()

        # State tracking
        self.current_step = 0
        self.aps = {}
        self.stations = {}
        self.interference_sources = []
        self.state_history = deque(maxlen=Config.STATE_HISTORY_LEN)

        # Metrics tracking with proper initialization
        self.step_throughput = []
        self.step_latency = []
        self.step_collision_rate = []
        self.step_fairness = []
        self.total_channel_switches = 0

        self.metrics = {
            'throughput': [],
            'latency': [],
            'collision_rate': [],
            'fairness': [],
            'channel_switches': 0
        }

        self._setup_action_observation_spaces()
        self._initialize_network()

    def _setup_action_observation_spaces(self):
        """Define action and observation spaces for each AP agent"""

        # Action space per AP: [channel_idx, channel_width_idx, tx_power_idx]
        self.action_space = spaces.Dict({
            f'ap_{i}': spaces.MultiDiscrete([
                len(Config.ALL_CHANNELS),    # Channel selection
                len(Config.CHANNEL_WIDTHS),  # Channel width
                len(Config.TX_POWER_LEVELS)  # TX power
            ]) for i in range(self.num_aps)
        })

        # Observation space per AP (local + partial global)
        obs_dim = (
            1 +  # Current channel
            1 +  # Current tx power
            1 +  # Queue length
            1 +  # Busy time fraction
            len(Config.ALL_CHANNELS) +  # Channel occupancy
            10 + # RSSI histogram bins
            5 +  # Traffic mix ratios
            3    # Neighboring AP interference levels
        )

        self.observation_space = spaces.Dict({
            f'ap_{i}': spaces.Box(
                low=-np.inf, high=np.inf,
                shape=(obs_dim,), dtype=np.float32
            ) for i in range(self.num_aps)
        })

    def _initialize_network(self):
        """Initialize APs, stations, and interference sources"""

        # Place APs in grid pattern
        ap_positions = self._generate_ap_positions()
        for i, pos in enumerate(ap_positions):
            self.aps[i] = AccessPoint(id=i, position=pos)

        # Place stations around each AP
        station_id = 0
        for ap_id in range(self.num_aps):
            ap_pos = self.aps[ap_id].position
            for _ in range(self.num_stations_per_ap):
                # Place stations in circle around AP
                angle = random.uniform(0, 2 * math.pi)
                distance = random.uniform(5, 20)  # 5-20m from AP
                sta_pos = (
                    ap_pos[0] + distance * math.cos(angle),
                    ap_pos[1] + distance * math.sin(angle)
                )

                traffic_type = random.choice(Config.TRAFFIC_TYPES)
                self.stations[station_id] = Station(
                    id=station_id,
                    position=sta_pos,
                    ap_id=ap_id,
                    traffic_type=traffic_type
                )
                self.aps[ap_id].stations.append(station_id)
                station_id += 1

        # Add external interference sources
        self._add_interference_sources()

        print(f"Initialized network: {self.num_aps} APs, {len(self.stations)} STAs")

    def _generate_ap_positions(self) -> List[Tuple[float, float]]:
        """Generate AP positions in realistic deployment patterns"""
        positions = []

        if self.config.get('topology', 'grid') == 'grid':
            # Grid topology for office/apartment scenarios
            rows = int(math.sqrt(self.num_aps))
            cols = math.ceil(self.num_aps / rows)
            spacing_x = self.area_size[0] / (cols + 1)
            spacing_y = self.area_size[1] / (rows + 1)

            for i in range(self.num_aps):
                row = i // cols
                col = i % cols
                x = (col + 1) * spacing_x
                y = (row + 1) * spacing_y
                # Add some randomness
                x += random.uniform(-spacing_x*0.2, spacing_x*0.2)
                y += random.uniform(-spacing_y*0.2, spacing_y*0.2)
                positions.append((x, y))

        return positions

    def _add_interference_sources(self):
        """Add external interference sources (Bluetooth, microwave, etc.)"""
        num_interferers = random.randint(1, 3)
        for _ in range(num_interferers):
            pos = (
                random.uniform(0, self.area_size[0]),
                random.uniform(0, self.area_size[1])
            )
            # Bluetooth-like interferer
            interferer = InterferenceSource(
                position=pos,
                power=random.uniform(0, 10),  # dBm
                frequency=2.4,  # GHz
                bandwidth=1,    # MHz
                duty_cycle=random.uniform(0.1, 0.8)
            )
            self.interference_sources.append(interferer)

    def reset(self, seed=None):
        """Reset environment to initial state"""
        if seed is not None:
            random.seed(seed)
            np.random.seed(seed)

        self.current_step = 0
        self.step_throughput = []
        self.step_latency = []
        self.step_collision_rate = []
        self.step_fairness = []
        self.total_channel_switches = 0

        self.metrics = {
            'throughput': [],
            'latency': [],
            'collision_rate': [],
            'fairness': [],
            'channel_switches': 0
        }

        # Reset AP states
        for ap in self.aps.values():
            ap.channel = random.choice(Config.CHANNELS_24GHZ)  # Start with 2.4GHz
            ap.queue_length = 0
            ap.busy_time = 0.0
            ap.collision_count = 0
            ap.throughput = 0.0
            ap.last_channel_switch = 0.0

        # Reset station states
        for sta in self.stations.values():
            sta.queue_length = random.randint(0, 5)
            sta.collision_count = 0
            sta.packets_sent = 0
            sta.packets_received = 0
            sta.total_delay = 0.0
            sta.last_transmission = 0.0

        self.state_history.clear()
        initial_obs = self._get_observations()
        initial_info = self._get_info()

        return initial_obs, initial_info

    def step(self, actions: Dict[str, np.ndarray]):
        """Execute one step of the environment"""

        # Apply actions (channel allocation, power control)
        self._apply_actions(actions)

        # Simulate MAC layer (CSMA/CA, collisions, throughput)
        self._simulate_mac_layer()

        # Calculate rewards
        rewards = self._calculate_rewards()

        # Update metrics
        self._update_metrics()

        self.current_step += 1
        terminated = self.current_step >= self.max_steps
        truncated = False

        observations = self._get_observations()
        info = self._get_info()

        return observations, rewards, terminated, truncated, info

    def _apply_actions(self, actions: Dict[str, np.ndarray]):
        """Apply agent actions to APs"""

        for agent_id, action in actions.items():
            ap_id = int(agent_id.split('_')[1])
            ap = self.aps[ap_id]

            # Track channel switches
            new_channel = Config.ALL_CHANNELS[action[0]]
            if new_channel != ap.channel:
                self.total_channel_switches += 1
                ap.last_channel_switch = self.current_step

            ap.channel = new_channel
            ap.channel_width = Config.CHANNEL_WIDTHS[action[1]]
            ap.tx_power = Config.TX_POWER_LEVELS[action[2]]

    def _simulate_mac_layer(self):
        """Simulate 802.11 MAC layer behavior with proper throughput calculation"""

        # Reset per-step metrics
        step_packets_sent = 0
        step_total_delay = 0.0
        step_collision_attempts = 0
        step_total_attempts = 0

        for ap in self.aps.values():
            ap.busy_time = 0.0
            ap.collision_count = 0
            ap.throughput = 0.0

        # Generate traffic
        self._generate_traffic()

        # Simulate multiple transmission opportunities per time step
        for _ in range(50):
            # Get transmission attempts
            transmission_attempts = self._get_transmission_attempts()
            step_total_attempts += len(transmission_attempts)

            # Resolve collisions
            successful_transmissions = self._resolve_collisions(transmission_attempts)

            # Count collisions
            step_collision_attempts += len(transmission_attempts) - len(successful_transmissions)

            # Update throughput and queues
            packets_this_round, delay_this_round = self._update_throughput_and_queues(successful_transmissions)
            step_packets_sent += packets_this_round
            step_total_delay += delay_this_round

        # Calculate step metrics
        step_throughput_mbps = self._calculate_step_throughput()
        step_latency_ms = step_total_delay / max(step_packets_sent, 1) * 1000  # Convert to ms
        step_collision_rate = step_collision_attempts / max(step_total_attempts, 1)
        step_fairness = self._calculate_fairness_index()

        # Store step metrics
        self.step_throughput.append(step_throughput_mbps)
        self.step_latency.append(step_latency_ms)
        self.step_collision_rate.append(step_collision_rate)
        self.step_fairness.append(step_fairness)

    def _calculate_step_throughput(self) -> float:
        """Calculate total network throughput for this step in Mbps"""
        total_throughput = 0.0

        for ap in self.aps.values():
            # Convert from bits to Mbps
            # throughput was accumulated in bits during transmission simulation
            total_throughput += ap.throughput

        return total_throughput

    def _generate_traffic(self):
        """Generate new packets based on traffic models"""
        for sta in self.stations.values():
            # Traffic arrival based on Poisson process
            arrival_rate = Config.ARRIVAL_RATES[sta.traffic_type]
            # Scale by time step duration
            arrival_prob = arrival_rate * self.time_step_duration

            # Generate multiple packets if needed
            packets_to_generate = np.random.poisson(arrival_prob)
            sta.queue_length += packets_to_generate

    def _get_transmission_attempts(self) -> List[Dict]:
        """Get all transmission attempts for this mini-slot"""
        attempts = []

        for sta in self.stations.values():
            if sta.queue_length > 0 and sta.backoff_slots <= 0:
                ap = self.aps[sta.ap_id]

                # Calculate transmission power
                tx_power = min(sta.tx_power, ap.tx_power)

                attempt = {
                    'station_id': sta.id,
                    'ap_id': sta.ap_id,
                    'channel': ap.channel,
                    'tx_power': tx_power,
                    'position': sta.position,
                    'packet_size': Config.PACKET_SIZES[sta.traffic_type],
                    'timestamp': self.current_step,
                    'traffic_type': sta.traffic_type
                }
                attempts.append(attempt)

                # Set new backoff
                cw = min(1023, 31 * (2 ** min(sta.collision_count, 10)))
                sta.backoff_slots = random.randint(0, cw)
            else:
                sta.backoff_slots = max(0, sta.backoff_slots - 1)

        return attempts

    def _resolve_collisions(self, attempts: List[Dict]) -> List[Dict]:
        """Resolve collisions and determine successful transmissions"""
        successful = []

        if not attempts:
            return successful

        # Group attempts by channel and AP
        channel_ap_attempts = {}
        for attempt in attempts:
            key = (attempt['channel'], attempt['ap_id'])
            if key not in channel_ap_attempts:
                channel_ap_attempts[key] = []
            channel_ap_attempts[key].append(attempt)

        # Check for collisions within each channel/AP group
        for (channel, ap_id), ch_attempts in channel_ap_attempts.items():
            if len(ch_attempts) == 1:
                # No collision
                successful.append(ch_attempts[0])
            else:
                # Collision detection using capture effect
                collision_resolved = self._capture_effect_resolution(ch_attempts)
                if collision_resolved:
                    successful.append(collision_resolved)
                else:
                    # All transmissions fail - update collision counts
                    for attempt in ch_attempts:
                        sta = self.stations[attempt['station_id']]
                        sta.collision_count += 1

        return successful

    def _capture_effect_resolution(self, colliding_attempts: List[Dict]) -> Optional[Dict]:
        """Resolve collisions using capture effect"""

        if len(colliding_attempts) < 2:
            return colliding_attempts[0] if colliding_attempts else None

        # Calculate received powers at the AP
        powers = []
        ap_id = colliding_attempts[0]['ap_id']  # All should have same AP
        ap = self.aps[ap_id]

        for attempt in colliding_attempts:
            rx_power = self.prop_model.received_power(
                attempt['tx_power'],
                attempt['position'],
                ap.position
            )
            powers.append((rx_power, attempt))

        # Sort by received power (highest first)
        powers.sort(key=lambda x: x[0], reverse=True)

        # Capture effect: strongest signal wins if > 10dB above next strongest
        if len(powers) > 1 and powers[0][0] - powers[1][0] > 10:
            return powers[0][1]
        else:
            return None  # Collision, no capture

    def _update_throughput_and_queues(self, successful_transmissions: List[Dict]) -> Tuple[int, float]:
        """Update throughput and queue states, return packets sent and total delay"""

        packets_sent = 0
        total_delay = 0.0

        for transmission in successful_transmissions:
            sta_id = transmission['station_id']
            sta = self.stations[sta_id]
            ap = self.aps[transmission['ap_id']]

            # Update station metrics
            if sta.queue_length > 0:
                sta.queue_length -= 1
                sta.packets_sent += 1
                packets_sent += 1

                # Calculate packet delay (queue time + transmission time)
                queue_delay = (self.current_step - sta.last_transmission) * self.time_step_duration

                # Calculate transmission time and data rate
                packet_size = transmission['packet_size']
                sinr = self._calculate_sinr(transmission)
                data_rate = self._sinr_to_rate(sinr)  # Mbps

                transmission_time = (packet_size * 8) / (data_rate * 1e6)  # seconds
                total_packet_delay = queue_delay + transmission_time

                sta.total_delay += total_packet_delay
                total_delay += total_packet_delay
                sta.last_transmission = self.current_step

                # Update AP metrics
                ap.busy_time += transmission_time / self.time_step_duration  # Fraction of step
                # Accumulate throughput in Mbps
                ap.throughput += (packet_size * 8) / (self.time_step_duration * 1e6)

        return packets_sent, total_delay

    def _calculate_sinr(self, transmission: Dict) -> float:
        """Calculate SINR for a transmission"""

        tx_pos = transmission['position']
        ap = self.aps[transmission['ap_id']]

        # Signal power
        signal_power = self.prop_model.received_power(
            transmission['tx_power'], tx_pos, ap.position
        )

        # Interference from other APs on same/adjacent channels
        interferer_powers = []
        for other_ap in self.aps.values():
            if other_ap.id != ap.id:
                separation_factor = self.interference_model.channel_separation_factor(
                    transmission['channel'], other_ap.channel
                )
                if separation_factor > 0.01:  # Significant interference
                    int_power = self.prop_model.received_power(
                        other_ap.tx_power, other_ap.position, ap.position
                    )
                    interferer_powers.append(int_power + 10*math.log10(separation_factor))

        return self.interference_model.calculate_sinr(signal_power, interferer_powers)

    def _sinr_to_rate(self, sinr_db: float) -> float:
        """Convert SINR to data rate (simplified 802.11 rate adaptation)"""
        if sinr_db >= 25:
            return 54.0    # 54 Mbps
        elif sinr_db >= 18:
            return 36.0    # 36 Mbps
        elif sinr_db >= 15:
            return 24.0    # 24 Mbps
        elif sinr_db >= 10:
            return 12.0    # 12 Mbps
        elif sinr_db >= 5:
            return 6.0     # 6 Mbps
        else:
            return 1.0     # 1 Mbps

    def _get_observations(self) -> Dict[str, np.ndarray]:
        """Get observations for all agents"""
        observations = {}

        for ap_id in range(self.num_aps):
            ap = self.aps[ap_id]
            obs = []

            # AP state
            obs.extend([
                ap.channel / 200.0,  # Normalized channel
                ap.tx_power / 30.0,  # Normalized TX power
                min(ap.queue_length / 100.0, 1.0),  # Normalized queue
                min(ap.busy_time, 1.0)  # Busy time fraction
            ])

            # Channel occupancy (interference levels)
            channel_occupancy = self._get_channel_occupancy(ap_id)
            obs.extend(channel_occupancy)

            # RSSI histogram from associated stations
            rssi_hist = self._get_rssi_histogram(ap_id)
            obs.extend(rssi_hist)

            # Traffic mix ratios
            traffic_mix = self._get_traffic_mix(ap_id)
            obs.extend(traffic_mix)

            # Neighboring AP interference
            neighbor_interference = self._get_neighbor_interference(ap_id)
            obs.extend(neighbor_interference)

            observations[f'ap_{ap_id}'] = np.array(obs, dtype=np.float32)

        return observations

    def _get_channel_occupancy(self, ap_id: int) -> List[float]:
        """Get normalized occupancy for each channel"""
        occupancy = [0.0] * len(Config.ALL_CHANNELS)

        for i, channel in enumerate(Config.ALL_CHANNELS):
            # Count APs using this channel
            count = sum(1 for ap in self.aps.values()
                       if ap.channel == channel and ap.id != ap_id)
            occupancy[i] = min(count / max(1, self.num_aps - 1), 1.0)

        return occupancy

    def _get_rssi_histogram(self, ap_id: int, bins: int = 10) -> List[float]:
        """Get RSSI histogram from associated stations"""
        ap = self.aps[ap_id]
        rssi_values = []

        for sta_id in ap.stations:
            sta = self.stations[sta_id]
            rssi = self.prop_model.received_power(
                sta.tx_power, sta.position, ap.position
            )
            rssi_values.append(rssi)

        if not rssi_values:
            return [0.0] * bins

        hist, _ = np.histogram(rssi_values, bins=bins, range=(-100, -30))
        return (hist / max(1, len(rssi_values))).tolist()

    def _get_traffic_mix(self, ap_id: int) -> List[float]:
        """Get traffic type distribution for AP"""
        ap = self.aps[ap_id]
        traffic_counts = {t: 0 for t in Config.TRAFFIC_TYPES}

        for sta_id in ap.stations:
            sta = self.stations[sta_id]
            traffic_counts[sta.traffic_type] += 1

        total = len(ap.stations) if ap.stations else 1
        return [traffic_counts[t] / total for t in Config.TRAFFIC_TYPES]

    def _get_neighbor_interference(self, ap_id: int) -> List[float]:
        """Get interference levels from neighboring APs"""
        ap = self.aps[ap_id]
        interference_levels = []

        # Find 3 closest APs
        distances = []
        for other_ap in self.aps.values():
            if other_ap.id != ap_id:
                dist = math.sqrt(
                    (ap.position[0] - other_ap.position[0])**2 +
                    (ap.position[1] - other_ap.position[1])**2
                )
                distances.append((dist, other_ap))

        distances.sort(key=lambda x: x[0])

        for i in range(3):  # Top 3 neighbors
            if i < len(distances):
                _, neighbor = distances[i]
                # Calculate interference level
                separation = self.interference_model.channel_separation_factor(
                    ap.channel, neighbor.channel
                )
                int_power = self.prop_model.received_power(
                    neighbor.tx_power, neighbor.position, ap.position
                )
                interference_levels.append(max(0, min(int_power / -30.0, 1.0)))  # Normalized
            else:
                interference_levels.append(0.0)

        return interference_levels

    def _calculate_rewards(self) -> Dict[str, float]:
        """Calculate per-agent rewards with proper scaling (0-1)"""
        rewards = {}

        # Get current step metrics
        current_throughput = self.step_throughput[-1] if self.step_throughput else 0
        current_latency = self.step_latency[-1] if self.step_latency else 0
        current_collision_rate = self.step_collision_rate[-1] if self.step_collision_rate else 0
        current_fairness = self.step_fairness[-1] if self.step_fairness else 1.0

        # Scale metrics to [0, 1] range
        # Throughput: normalize by target throughput (adjust based on network size)
        target_throughput = self.num_aps * 2  # 10 Mbps per AP target
        throughput_norm = min(current_throughput / target_throughput, 1.0)

        # Latency: invert and normalize (target < 10ms, bad > 100ms)
        latency_norm = max(0, 1.0 - min(current_latency / 100.0, 1.0))

        # Collision rate: invert (0 = good, 1 = bad)
        collision_norm = 1.0 - min(current_collision_rate, 1.0)

        # Fairness is already 0-1 (Jain's index)
        fairness_norm = current_fairness

        # Switching penalty
        recent_switches = sum(1 for ap in self.aps.values()
                            if self.current_step - ap.last_channel_switch < 10)
        switching_norm = 1.0 - min(recent_switches / self.num_aps, 1.0)

        # Composite reward
        base_reward = (
            Config.REWARD_WEIGHTS['throughput'] * throughput_norm +
            Config.REWARD_WEIGHTS['latency'] * latency_norm +
            Config.REWARD_WEIGHTS['collision'] * collision_norm +
            Config.REWARD_WEIGHTS['switching'] * switching_norm +
            Config.REWARD_WEIGHTS['fairness'] * fairness_norm
        )

        # Per-AP rewards with local performance bonus
        total_ap_throughput = sum(ap.throughput for ap in self.aps.values())

        for ap_id in range(self.num_aps):
            ap = self.aps[ap_id]
            if total_ap_throughput > 0:
                local_bonus = ap.throughput / total_ap_throughput
            else:
                local_bonus = 0.0
            rewards[f'ap_{ap_id}'] = base_reward + 0.1 * local_bonus

        return rewards

    def _calculate_average_latency(self) -> float:
        """Calculate average packet latency in ms"""
        if not self.step_latency:
            return 0.0
        return np.mean(self.step_latency[-100:]) if len(self.step_latency) >= 100 else np.mean(self.step_latency)

    def _calculate_collision_rate(self) -> float:
        """Calculate overall collision rate"""
        if not self.step_collision_rate:
            return 0.0
        return np.mean(self.step_collision_rate[-100:]) if len(self.step_collision_rate) >= 100 else np.mean(self.step_collision_rate)

    def _calculate_fairness_index(self) -> float:
        """Calculate Jain's fairness index"""
        throughputs = [ap.throughput for ap in self.aps.values()]
        if not throughputs or all(t == 0 for t in throughputs):
            return 1.0

        sum_x = sum(throughputs)
        sum_x_squared = sum(x**2 for x in throughputs)
        n = len(throughputs)

        if sum_x_squared == 0:
            return 1.0

        return (sum_x**2) / (n * sum_x_squared)

    def _update_metrics(self):
        """Update environment metrics for monitoring"""
        # Store current step metrics
        current_throughput = self.step_throughput[-1] if self.step_throughput else 0
        current_latency = self.step_latency[-1] if self.step_latency else 0
        current_collision_rate = self.step_collision_rate[-1] if self.step_collision_rate else 0
        current_fairness = self.step_fairness[-1] if self.step_fairness else 1.0

        self.metrics['throughput'].append(current_throughput)
        self.metrics['latency'].append(current_latency)
        self.metrics['collision_rate'].append(current_collision_rate)
        self.metrics['fairness'].append(current_fairness)
        self.metrics['channel_switches'] = self.total_channel_switches

    def _get_info(self) -> Dict:
        """Get additional environment information"""
        return {
            'step': self.current_step,
            'metrics': self.metrics.copy(),
            'network_state': {
                'aps': {ap_id: {
                    'channel': ap.channel,
                    'tx_power': ap.tx_power,
                    'throughput': ap.throughput,
                    'stations': len(ap.stations),
                    'position': ap.position
                } for ap_id, ap in self.aps.items()},
                'total_stations': len(self.stations),
                'interference_sources': len(self.interference_sources)
            }
        }

# GRAPH NEURAL NETWORK POLICY (with proper edge features)

In [ ]:
class GNNWiFiPolicy(nn.Module):
    """GNN policy with proper edge feature handling"""

    def __init__(self, node_features: int, edge_features: int, hidden_dim: int = 128,
                 num_channels: int = len(Config.ALL_CHANNELS),
                 num_powers: int = len(Config.TX_POWER_LEVELS),
                 num_widths: int = len(Config.CHANNEL_WIDTHS)):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.num_channels = num_channels
        self.num_powers = num_powers
        self.num_widths = num_widths

        # Node embedding layers
        self.node_encoder = nn.Sequential(
            nn.Linear(node_features, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        # Edge embedding for edge features
        self.edge_encoder = nn.Sequential(
            nn.Linear(edge_features, hidden_dim // 4),
            nn.ReLU()
        )

        # Graph attention layers with edge features
        self.gat_layers = nn.ModuleList([
            GATConv(hidden_dim, hidden_dim // 4, heads=4, dropout=0.1, edge_dim=hidden_dim // 4)
            for _ in range(3)
        ])

        # LSTM for temporal modeling
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)

        # Action heads
        self.channel_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, num_channels)
        )

        self.power_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, num_powers)
        )

        self.width_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, num_widths)
        )

        # Value head for critic
        self.value_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, graph_batch, hidden_state=None):
        """Forward pass through GNN policy"""

        # Node encoding
        x = self.node_encoder(graph_batch.x)

        # Edge encoding
        edge_attr = self.edge_encoder(graph_batch.edge_attr) if graph_batch.edge_attr.size(0) > 0 else None

        # Graph attention layers with edge features
        for gat_layer in self.gat_layers:
            if edge_attr is not None:
                x = gat_layer(x, graph_batch.edge_index, edge_attr=edge_attr)
            else:
                x = gat_layer(x, graph_batch.edge_index)
            x = F.relu(x)

        # Global pooling to get graph-level representation
        graph_repr = global_mean_pool(x, graph_batch.batch)

        # Temporal modeling with LSTM
        if hidden_state is not None:
            lstm_out, new_hidden = self.lstm(graph_repr.unsqueeze(1), hidden_state)
            features = lstm_out.squeeze(1)
        else:
            features = graph_repr
            new_hidden = None

        # Action predictions
        channel_logits = self.channel_head(features)
        power_logits = self.power_head(features)
        width_logits = self.width_head(features)

        # Value prediction
        values = self.value_head(features)

        return {
            'channel_logits': channel_logits,
            'power_logits': power_logits,
            'width_logits': width_logits,
            'values': values,
            'hidden_state': new_hidden
        }

In [ ]:
class WiFiGraphBuilder:
    """Builds graph representations from WiFi environment state"""

    def __init__(self, max_range: float = 50.0):
        self.max_range = max_range

    def build_graph(self, env_state: Dict) -> Data:
        """Build PyTorch Geometric graph from environment state"""

        # Extract AP and station positions and features
        ap_nodes = []
        ap_positions = []

        for ap_id, ap_info in env_state['aps'].items():
            # AP node features: [channel, tx_power, throughput, num_stations]
            features = [
                ap_info['channel'] / 200.0,  # Normalized
                ap_info['tx_power'] / 30.0,
                ap_info['throughput'] / 50.0,  # Target throughput normalization
                ap_info['stations'] / 10.0,  # Max expected stations
            ]
            ap_nodes.append(features)
            ap_positions.append(ap_info['position'])

        # Build edge indices based on interference range
        edge_indices = []
        edge_features = []

        for i, pos_i in enumerate(ap_positions):
            for j, pos_j in enumerate(ap_positions):
                if i != j:
                    distance = math.sqrt((pos_i[0] - pos_j[0])**2 + (pos_i[1] - pos_j[1])**2)
                    if distance <= self.max_range:
                        edge_indices.append([i, j])

                        # Edge features: [distance, channel_separation, power_difference]
                        ch_i = env_state['aps'][str(i)]['channel']
                        ch_j = env_state['aps'][str(j)]['channel']
                        power_i = env_state['aps'][str(i)]['tx_power']
                        power_j = env_state['aps'][str(j)]['tx_power']

                        edge_feat = [
                            distance / self.max_range,  # Normalized distance
                            abs(ch_i - ch_j) / 200.0,  # Channel separation
                            abs(power_i - power_j) / 20.0  # Power difference
                        ]
                        edge_features.append(edge_feat)

        # Convert to tensors
        node_features = torch.FloatTensor(ap_nodes)
        edge_index = torch.LongTensor(edge_indices).t().contiguous() if edge_indices else torch.empty(2, 0, dtype=torch.long)
        edge_attr = torch.FloatTensor(edge_features) if edge_features else torch.empty(0, 3)

        return Data(x=node_features, edge_index=edge_index, edge_attr=edge_attr)

# MAPPO TRAINING ALGORITHM (with fixed value clipping)

In [ ]:
class MAPPOAgent:
    """Multi-Agent PPO with Centralized Training, Decentralized Execution - Fixed Version"""

    def __init__(self, env, config: Dict, logger: CSVLogger):
        self.env = env
        self.config = config
        self.logger = logger
        self.device = torch.device('cuda:3' if torch.cuda.is_available() else 'cpu')

        # Network parameters
        self.node_features = 4  # AP features
        self.edge_features = 3  # Edge features
        self.hidden_dim = config.get('hidden_dim', 128)

        # Training parameters
        self.learning_rate = config.get('learning_rate', 3e-4)
        self.gamma = config.get('gamma', 0.99)
        self.gae_lambda = config.get('gae_lambda', 0.95)
        self.clip_ratio = config.get('clip_ratio', 0.2)
        self.value_clip = config.get('value_clip', True)
        self.entropy_coef = config.get('entropy_coef', 0.01)
        self.value_coef = config.get('value_coef', 0.25)  # Reduced from 0.5 to lower value loss
        self.max_grad_norm = config.get('max_grad_norm', 0.5)

        # Initialize networks
        self.policy_net = GNNWiFiPolicy(
            self.node_features,
            self.edge_features,
            self.hidden_dim
        ).to(self.device)

        self.optimizer = torch.optim.Adam(
            self.policy_net.parameters(),
            lr=self.learning_rate
        )

        # Graph builder
        self.graph_builder = WiFiGraphBuilder()

        # Training metrics
        self.training_metrics = {
            'episode_rewards': [],
            'policy_loss': [],
            'value_loss': [],
            'entropy_loss': []
        }

        # Model checkpointing
        self.best_reward = float('-inf')
        self.update_count = 0

    def collect_trajectories(self, num_steps: int):
        """Collect training trajectories"""

        trajectories = []
        obs, info = self.env.reset()

        episode_rewards = {agent: 0 for agent in obs.keys()}
        episode_steps = 0
        episode_count = 0

        for step in range(num_steps):
            # Convert observations to graph
            graph_data = self._obs_to_graph(obs, info)

            # Get actions from policy
            with torch.no_grad():
                actions, log_probs, values = self._get_actions_and_values(graph_data)

            # Environment step
            next_obs, rewards, terminated, truncated, next_info = self.env.step(actions)

            # Store trajectory data
            trajectories.append({
                'obs': obs.copy(),
                'actions': actions.copy(),
                'rewards': rewards.copy(),
                'log_probs': log_probs,
                'values': values,
                'info': info.copy()
            })

            # Update episode tracking
            for agent, reward in rewards.items():
                episode_rewards[agent] += reward
            episode_steps += 1

            obs = next_obs
            info = next_info

            if terminated or truncated:
                # Log episode metrics
                avg_episode_reward = sum(episode_rewards.values()) / len(episode_rewards)
                self.training_metrics['episode_rewards'].append(avg_episode_reward)

                # Extract metrics from final info
                metrics = info['metrics']
                episode_metrics = {
                    'total_reward': avg_episode_reward,
                    'avg_throughput': np.mean(metrics['throughput'][-100:]) if metrics['throughput'] else 0,
                    'avg_latency': np.mean(metrics['latency'][-100:]) if metrics['latency'] else 0,
                    'avg_collision_rate': np.mean(metrics['collision_rate'][-100:]) if metrics['collision_rate'] else 0,
                    'avg_fairness': np.mean(metrics['fairness'][-100:]) if metrics['fairness'] else 0,
                }

                self.logger.log_episode(episode_count, episode_steps, episode_metrics)
                print(f"Episode {episode_count}: reward={avg_episode_reward:.3f}, "
                      f"throughput={episode_metrics['avg_throughput']:.2f} Mbps, "
                      f"latency={episode_metrics['avg_latency']:.1f} ms, steps={episode_steps}")

                # Reset environment
                obs, info = self.env.reset()
                episode_rewards = {agent: 0 for agent in obs.keys()}
                episode_steps = 0
                episode_count += 1

        return trajectories

    def _obs_to_graph(self, obs: Dict, info: Dict) -> Data:
        """Convert multi-agent observations to graph representation"""

        # Extract AP information from observations and info
        ap_info = {}
        for agent_id, agent_obs in obs.items():
            ap_id = int(agent_id.split('_')[1])

            # Decode observation features (this should match _get_observations)
            ap_info[str(ap_id)] = {
                'channel': agent_obs[0] * 200.0,  # Denormalize
                'tx_power': agent_obs[1] * 30.0,
                'throughput': info['network_state']['aps'][ap_id]['throughput'],
                'stations': info['network_state']['aps'][ap_id]['stations'],
                'position': info['network_state']['aps'][ap_id]['position']
            }

        env_state = {'aps': ap_info}
        return self.graph_builder.build_graph(env_state)

    def _get_actions_and_values(self, graph_data: Data):
        """Get actions and values from policy network"""

        graph_batch = Batch.from_data_list([graph_data]).to(self.device)

        # Forward pass
        output = self.policy_net(graph_batch)

        # Sample actions
        channel_dist = torch.distributions.Categorical(logits=output['channel_logits'])
        power_dist = torch.distributions.Categorical(logits=output['power_logits'])
        width_dist = torch.distributions.Categorical(logits=output['width_logits'])

        channel_actions = channel_dist.sample()
        power_actions = power_dist.sample()
        width_actions = width_dist.sample()

        # Calculate log probabilities
        channel_log_probs = channel_dist.log_prob(channel_actions)
        power_log_probs = power_dist.log_prob(power_actions)
        width_log_probs = width_dist.log_prob(width_actions)

        # Combine actions and log probs
        actions = {}
        log_probs = {}
        values = {}

        for i in range(len(channel_actions)):
            agent_id = f'ap_{i}'
            actions[agent_id] = np.array([
                channel_actions[i].item(),
                width_actions[i].item(),
                power_actions[i].item()
            ])
            log_probs[agent_id] = (
                channel_log_probs[i] +
                power_log_probs[i] +
                width_log_probs[i]
            ).item()
            values[agent_id] = output['values'][i].item()

        return actions, log_probs, values

    def update_policy(self, trajectories: List[Dict]):
        """Update policy using PPO algorithm with improved value loss handling"""

        # Calculate advantages using GAE
        processed_trajectories = self._calculate_advantages(trajectories)

        # Convert to tensors
        all_graphs = []
        all_actions = []
        all_old_log_probs = []
        all_advantages = []
        all_returns = []
        all_old_values = []

        for traj in processed_trajectories:
            graph_data = self._obs_to_graph(traj['obs'], traj['info'])
            all_graphs.append(graph_data)

            # Flatten multi-agent data
            actions = []
            old_log_probs = []
            advantages = []
            returns = []
            old_values = []

            for agent_id in sorted(traj['actions'].keys()):
                actions.append(traj['actions'][agent_id])
                old_log_probs.append(traj['log_probs'][agent_id])
                advantages.append(traj['advantages'][agent_id])
                returns.append(traj['returns'][agent_id])
                old_values.append(traj['values'][agent_id])

            all_actions.append(np.array(actions))
            all_old_log_probs.append(np.array(old_log_probs))
            all_advantages.append(np.array(advantages))
            all_returns.append(np.array(returns))
            all_old_values.append(np.array(old_values))

        # Convert to tensors
        graph_batch = Batch.from_data_list(all_graphs).to(self.device)
        actions_tensor = torch.FloatTensor(np.array(all_actions)).to(self.device)
        old_log_probs_tensor = torch.FloatTensor(np.array(all_old_log_probs)).to(self.device)
        advantages_tensor = torch.FloatTensor(np.array(all_advantages)).to(self.device)
        returns_tensor = torch.FloatTensor(np.array(all_returns)).to(self.device)
        old_values_tensor = torch.FloatTensor(np.array(all_old_values)).to(self.device)

        # Normalize advantages
        advantages_tensor = (advantages_tensor - advantages_tensor.mean()) / (advantages_tensor.std() + 1e-8)

        # PPO update with multiple epochs
        total_losses = []
        for epoch in range(3):  # Reduced from 4 to 3 epochs
            # Forward pass
            output = self.policy_net(graph_batch)

            # Calculate new log probabilities
            channel_dist = torch.distributions.Categorical(logits=output['channel_logits'])
            power_dist = torch.distributions.Categorical(logits=output['power_logits'])
            width_dist = torch.distributions.Categorical(logits=output['width_logits'])

            new_log_probs = (
                channel_dist.log_prob(actions_tensor[:, :, 0].long()) +
                power_dist.log_prob(actions_tensor[:, :, 2].long()) +
                width_dist.log_prob(actions_tensor[:, :, 1].long())
            ).sum(dim=1)

            # Calculate ratio and clipped objective
            ratio = torch.exp(new_log_probs - old_log_probs_tensor.sum(dim=1))
            surr1 = ratio * advantages_tensor.sum(dim=1)
            surr2 = torch.clamp(ratio, 1 - self.clip_ratio, 1 + self.clip_ratio) * advantages_tensor.sum(dim=1)

            policy_loss = -torch.min(surr1, surr2).mean()

            # Improved value loss with gradient clipping and normalization
            values = output['values'].squeeze()
            returns_sum = returns_tensor.sum(dim=1)
            old_values_sum = old_values_tensor.sum(dim=1)

            # Normalize returns for better value function training
            returns_normalized = (returns_sum - returns_sum.mean()) / (returns_sum.std() + 1e-8)
            values_normalized = (values - values.mean()) / (values.std() + 1e-8)

            if self.value_clip:
                # Clipped value loss with normalized values
                values_clipped = old_values_sum + torch.clamp(
                    values - old_values_sum,
                    -self.clip_ratio,
                    self.clip_ratio
                )
                value_loss_1 = (values_normalized - returns_normalized) ** 2
                value_loss_2 = ((values_clipped - values_clipped.mean()) / (values_clipped.std() + 1e-8) - returns_normalized) ** 2
                value_loss = torch.max(value_loss_1, value_loss_2).mean()
            else:
                value_loss = ((values_normalized - returns_normalized) ** 2).mean()

            # Entropy loss
            entropy = (channel_dist.entropy() + power_dist.entropy() + width_dist.entropy()).mean()

            # Total loss with reduced value coefficient
            total_loss = policy_loss + self.value_coef * value_loss - self.entropy_coef * entropy

            # Optimization step
            self.optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), self.max_grad_norm)
            self.optimizer.step()

            total_losses.append(total_loss.item())

        # Log training metrics
        losses = {
            'policy_loss': policy_loss.item(),
            'value_loss': value_loss.item(),
            'entropy_loss': entropy.item(),
            'total_loss': np.mean(total_losses)
        }

        self.logger.log_training(self.update_count, losses)
        self.training_metrics['policy_loss'].append(policy_loss.item())
        self.training_metrics['value_loss'].append(value_loss.item())
        self.training_metrics['entropy_loss'].append(entropy.item())

        self.update_count += 1
        print(f"Update {self.update_count}: policy_loss={policy_loss.item():.4f}, "
              f"value_loss={value_loss.item():.4f}, entropy={entropy.item():.4f}")

    def _calculate_advantages(self, trajectories: List[Dict]) -> List[Dict]:
        """Calculate advantages using Generalized Advantage Estimation"""

        # Organize data by agent across all trajectories
        agent_data = {}

        # Get all agent IDs from first trajectory
        if not trajectories:
            return []

        agent_ids = list(trajectories[0]['rewards'].keys())

        # Initialize agent data storage
        for agent_id in agent_ids:
            agent_data[agent_id] = {
                'rewards': [],
                'values': [],
                'trajectory_indices': []
            }

        # Collect data for each agent across trajectories
        for traj_idx, traj in enumerate(trajectories):
            for agent_id in agent_ids:
                if agent_id in traj['rewards'] and agent_id in traj['values']:
                    agent_data[agent_id]['rewards'].append(traj['rewards'][agent_id])
                    agent_data[agent_id]['values'].append(traj['values'][agent_id])
                    agent_data[agent_id]['trajectory_indices'].append(traj_idx)

        # Calculate advantages for each agent
        agent_advantages = {}
        agent_returns = {}

        for agent_id in agent_ids:
            rewards = agent_data[agent_id]['rewards']
            values = agent_data[agent_id]['values']

            if not rewards or not values:
                continue

            # GAE calculation
            advantages = []
            returns = []

            gae = 0
            for t in reversed(range(len(rewards))):
                if t == len(rewards) - 1:
                    next_value = 0  # Terminal value
                else:
                    next_value = values[t + 1]

                delta = rewards[t] + self.gamma * next_value - values[t]
                gae = delta + self.gamma * self.gae_lambda * gae
                advantages.insert(0, gae)
                returns.insert(0, gae + values[t])

            agent_advantages[agent_id] = advantages
            agent_returns[agent_id] = returns

        # Map advantages and returns back to trajectories
        processed_trajectories = []

        for traj_idx, traj in enumerate(trajectories):
            enhanced_traj = traj.copy()
            traj_advantages = {}
            traj_returns = {}

            for agent_id in agent_ids:
                if agent_id in agent_advantages:
                    agent_traj_indices = agent_data[agent_id]['trajectory_indices']
                    if traj_idx in agent_traj_indices:
                        advantage_idx = agent_traj_indices.index(traj_idx)
                        if advantage_idx < len(agent_advantages[agent_id]):
                            traj_advantages[agent_id] = agent_advantages[agent_id][advantage_idx]
                            traj_returns[agent_id] = agent_returns[agent_id][advantage_idx]
                        else:
                            traj_advantages[agent_id] = 0.0
                            traj_returns[agent_id] = 0.0
                    else:
                        traj_advantages[agent_id] = 0.0
                        traj_returns[agent_id] = 0.0
                else:
                    traj_advantages[agent_id] = 0.0
                    traj_returns[agent_id] = 0.0

            enhanced_traj['advantages'] = traj_advantages
            enhanced_traj['returns'] = traj_returns
            processed_trajectories.append(enhanced_traj)

        return processed_trajectories

    def train(self, total_timesteps: int, steps_per_update: int = 2048):
        """Main training loop"""

        print(f"Starting MAPPO training: {total_timesteps} timesteps")

        timesteps_collected = 0

        while timesteps_collected < total_timesteps:
            # Collect trajectories
            trajectories = self.collect_trajectories(steps_per_update)
            timesteps_collected += len(trajectories)

            # Update policy
            self.update_policy(trajectories)

            # Save checkpoints
            self._save_checkpoint()

            # Periodic evaluation
            if self.update_count % 10 == 0:
                avg_reward = self.evaluate()
                if avg_reward > self.best_reward:
                    self.best_reward = avg_reward
                    self._save_best_model()

        print("Training completed!")

    def evaluate(self, num_episodes: int = 5) -> float:
        """Evaluate current policy"""

        episode_rewards = []
        episode_metrics = {
            'throughput': [],
            'latency': [],
            'collision_rate': [],
            'fairness': []
        }

        for episode in range(num_episodes):
            obs, info = self.env.reset()
            episode_reward = 0
            done = False
            step_count = 0

            while not done and step_count < self.env.max_steps:
                # Get actions (without exploration)
                graph_data = self._obs_to_graph(obs, info)
                with torch.no_grad():
                    actions, _, _ = self._get_actions_and_values(graph_data)

                obs, rewards, terminated, truncated, info = self.env.step(actions)
                episode_reward += sum(rewards.values())
                done = terminated or truncated
                step_count += 1

            episode_rewards.append(episode_reward)

            # Collect metrics from final state
            metrics = info['metrics']
            if metrics['throughput']:
                episode_metrics['throughput'].append(np.mean(metrics['throughput'][-100:]))
            if metrics['latency']:
                episode_metrics['latency'].append(np.mean(metrics['latency'][-100:]))
            if metrics['collision_rate']:
                episode_metrics['collision_rate'].append(np.mean(metrics['collision_rate'][-100:]))
            if metrics['fairness']:
                episode_metrics['fairness'].append(np.mean(metrics['fairness'][-100:]))

        # Log evaluation results
        avg_reward = np.mean(episode_rewards)
        avg_throughput = np.mean(episode_metrics['throughput']) if episode_metrics['throughput'] else 0
        avg_latency = np.mean(episode_metrics['latency']) if episode_metrics['latency'] else 0
        avg_collision = np.mean(episode_metrics['collision_rate']) if episode_metrics['collision_rate'] else 0
        avg_fairness = np.mean(episode_metrics['fairness']) if episode_metrics['fairness'] else 0

        print(f"Evaluation Results:")
        print(f"  Average Reward: {avg_reward:.3f}")
        print(f"  Average Throughput: {avg_throughput:.2f} Mbps")
        print(f"  Average Latency: {avg_latency:.2f} ms")
        print(f"  Average Collision Rate: {avg_collision:.3f}")
        print(f"  Average Fairness: {avg_fairness:.3f}")

        return avg_reward

    def _save_checkpoint(self):
        """Save latest checkpoint (overwrite)"""
        checkpoint = {
            'model_state_dict': self.policy_net.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'training_config': self.config,
            'training_metrics': self.training_metrics,
            'update_count': self.update_count,
            'best_reward': self.best_reward
        }
        torch.save(checkpoint, '/content/drive/MyDrive/WiFi_Generalization/wifi_rl_logs/wifi_rl_checkpoint.pth')

    def _save_best_model(self):
        """Save best model (overwrite)"""
        best_model = {
            'model_state_dict': self.policy_net.state_dict(),
            'training_config': self.config,
            'best_reward': self.best_reward,
            'update_count': self.update_count
        }
        torch.save(best_model, '/content/drive/MyDrive/WiFi_Generalization/wifi_rl_logs/wifi_rl_best_model.pth')
        print(f"New best model saved with reward: {self.best_reward:.3f}")

# BASELINES AND EVALUATION

In [ ]:
class BaselineAgent:
    """Baseline agents for comparison"""

    @staticmethod
    def random_allocation(env):
        """Random channel allocation baseline"""
        actions = {}
        for ap_id in range(env.num_aps):
            actions[f'ap_{ap_id}'] = np.array([
                random.randint(0, len(Config.ALL_CHANNELS) - 1),
                random.randint(0, len(Config.CHANNEL_WIDTHS) - 1),
                random.randint(0, len(Config.TX_POWER_LEVELS) - 1)
            ])
        return actions

    @staticmethod
    def least_congested_channel(env):
        """Least congested channel baseline"""
        # Count channel usage
        channel_usage = {ch: 0 for ch in Config.ALL_CHANNELS}
        for ap in env.aps.values():
            channel_usage[ap.channel] += 1

        actions = {}
        for ap_id in range(env.num_aps):
            # Find least used channel
            min_usage = min(channel_usage.values())
            available_channels = [ch for ch, usage in channel_usage.items() if usage == min_usage]
            selected_channel = random.choice(available_channels)

            channel_idx = Config.ALL_CHANNELS.index(selected_channel)
            actions[f'ap_{ap_id}'] = np.array([channel_idx, 0, 2])  # 20MHz, medium power
            channel_usage[selected_channel] += 1

        return actions

In [ ]:
class EvaluationSuite:
    """Comprehensive evaluation suite with corrected sequential action execution"""

    def __init__(self, env, trained_agent=None, logger=None):
        self.env = env
        self.trained_agent = trained_agent
        self.logger = logger

    def run_comparison(self, num_episodes: int = 50):
        """Run comparison between trained agent and baselines"""

        results = {}

        # Test trained agent
        if self.trained_agent:
            print("Evaluating trained MAPPO agent...")
            results['MAPPO'] = self._evaluate_agent_sequential(
                self._get_trained_actions, num_episodes
            )

        # Test baselines
        print("Evaluating random baseline...")
        results['Random'] = self._evaluate_agent_sequential(
            lambda obs, info: BaselineAgent.random_allocation(self.env), num_episodes
        )

        print("Evaluating least congested baseline...")
        results['Least Congested'] = self._evaluate_agent_sequential(
            lambda obs, info: BaselineAgent.least_congested_channel(self.env), num_episodes
        )

        return results

    def _evaluate_agent_sequential(self, action_fn, num_episodes: int):
        """Evaluate a single agent with proper sequential execution"""

        episode_metrics = []

        for episode in range(num_episodes):
            obs, info = self.env.reset()
            episode_data = {
                'total_reward': 0,
                'throughput': [],
                'latency': [],
                'collision_rate': [],
                'fairness': [],
                'channel_switches': 0
            }

            step_count = 0
            while step_count < self.env.max_steps:
                # Get actions using the provided function
                actions = action_fn(obs, info)

                # Execute step
                obs, rewards, terminated, truncated, info = self.env.step(actions)

                episode_data['total_reward'] += sum(rewards.values())

                # Collect step metrics
                metrics = info['metrics']
                if metrics['throughput']:
                    episode_data['throughput'].append(metrics['throughput'][-1])
                if metrics['latency']:
                    episode_data['latency'].append(metrics['latency'][-1])
                if metrics['collision_rate']:
                    episode_data['collision_rate'].append(metrics['collision_rate'][-1])
                if metrics['fairness']:
                    episode_data['fairness'].append(metrics['fairness'][-1])

                episode_data['channel_switches'] = metrics['channel_switches']

                step_count += 1
                if terminated or truncated:
                    break

            # Calculate episode averages
            episode_summary = {
                'total_reward': episode_data['total_reward'],
                'avg_throughput': np.mean(episode_data['throughput']) if episode_data['throughput'] else 0,
                'avg_latency': np.mean(episode_data['latency']) if episode_data['latency'] else 0,
                'avg_collision_rate': np.mean(episode_data['collision_rate']) if episode_data['collision_rate'] else 0,
                'avg_fairness': np.mean(episode_data['fairness']) if episode_data['fairness'] else 0,
                'channel_switches': episode_data['channel_switches']
            }
            episode_metrics.append(episode_summary)

            # Log to CSV if logger available
            if self.logger:
                agent_name = "MAPPO" if action_fn == self._get_trained_actions else "Baseline"
                self.logger.log_evaluation(agent_name, episode, episode_summary)

        # Calculate overall statistics
        return {
            'mean_reward': np.mean([ep['total_reward'] for ep in episode_metrics]),
            'std_reward': np.std([ep['total_reward'] for ep in episode_metrics]),
            'mean_throughput': np.mean([ep['avg_throughput'] for ep in episode_metrics]),
            'mean_latency': np.mean([ep['avg_latency'] for ep in episode_metrics]),
            'mean_collision_rate': np.mean([ep['avg_collision_rate'] for ep in episode_metrics]),
            'mean_fairness': np.mean([ep['avg_fairness'] for ep in episode_metrics]),
            'total_channel_switches': np.sum([ep['channel_switches'] for ep in episode_metrics])
        }

    def _get_trained_actions(self, obs, info):
        """Get actions from trained agent"""
        graph_data = self.trained_agent._obs_to_graph(obs, info)
        with torch.no_grad():
            actions, _, _ = self.trained_agent._get_actions_and_values(graph_data)
        return actions

    def plot_results(self, results: Dict, save_path: str = 'wifi_rl_evaluation_results.png'):
        """Plot evaluation results"""

        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        metrics = ['mean_throughput', 'mean_latency', 'mean_collision_rate',
                  'mean_fairness', 'total_channel_switches', 'mean_reward']
        titles = ['Average Throughput (Mbps)', 'Average Latency (ms)',
                 'Average Collision Rate', 'Average Fairness Index',
                 'Total Channel Switches', 'Average Reward']

        for i, (metric, title) in enumerate(zip(metrics, titles)):
            ax = axes[i // 3, i % 3]

            agents = list(results.keys())
            values = [results[agent][metric] for agent in agents]

            bars = ax.bar(agents, values)
            ax.set_title(title)
            ax.set_ylabel('Value')

            # Color bars
            colors = ['green', 'orange', 'blue']
            for bar, color in zip(bars, colors[:len(bars)]):
                bar.set_color(color)

            # Add value labels on bars
            for bar, value in zip(bars, values):
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height,
                       f'{value:.2f}', ha='center', va='bottom')

        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"Results plot saved to {save_path}")

# SCENARIO GENERATOR

In [ ]:
class ScenarioGenerator:
    """Generate diverse testing scenarios"""

    @staticmethod
    def dense_apartment_scenario():
        """Dense apartment building scenario"""
        return {
            'num_aps': 6,
            'stations_per_ap': 8,
            'area_size': (60, 60),
            'topology': 'grid',
            'max_steps': 1000,
            'interference_level': 'high',
            'mobility': 'low'
        }

    @staticmethod
    def office_scenario():
        """Office building scenario"""
        return {
            'num_aps': 4,
            'stations_per_ap': 10,
            'area_size': (80, 80),
            'topology': 'grid',
            'max_steps': 800,
            'interference_level': 'medium',
            'mobility': 'medium'
        }

    @staticmethod
    def hotspot_scenario():
        """High-density hotspot scenario"""
        return {
            'num_aps': 8,
            'stations_per_ap': 15,
            'area_size': (40, 40),
            'topology': 'clustered',
            'max_steps': 600,
            'interference_level': 'very_high',
            'mobility': 'high'
        }

# MAIN TRAINING AND EVALUATION PIPELINE

In [ ]:
def main_training_pipeline():
    """Main pipeline for training and evaluating WiFi RL agents"""

    print("Starting WiFi RL Testbed Pipeline")

    # Initialize CSV logger
    logger = CSVLogger()

    # Training configuration
    training_config = {
        'hidden_dim': 128,
        'learning_rate': 3e-4,
        'gamma': 0.99,
        'gae_lambda': 0.95,
        'clip_ratio': 0.2,
        'entropy_coef': 0.005,
        'value_coef': 0.5,
        'max_grad_norm': 0.5
    }

    # 1. Create scenarios for curriculum learning
    scenarios = [
        ScenarioGenerator.office_scenario(),
        ScenarioGenerator.dense_apartment_scenario(),
        ScenarioGenerator.hotspot_scenario()
    ]

    print(f"Created {len(scenarios)} training scenarios")

    # 2. Initialize environment and agent
    env = WiFiEnvironment(scenarios[0])  # Start with office scenario
    agent = MAPPOAgent(env, training_config, logger)

    # 3. Curriculum training
    for i, scenario in enumerate(scenarios):
        print(f"Training on scenario {i+1}/{len(scenarios)}: {scenario}")

        # Update environment with new scenario
        env = WiFiEnvironment(scenario)
        agent.env = env

        # Train for this scenario
        timesteps_per_scenario = 100000 // len(scenarios)
        agent.train(timesteps_per_scenario, steps_per_update=2048)

    # 4. Final evaluation
    print("Running final evaluation...")
    evaluator = EvaluationSuite(env, agent, logger)
    results = evaluator.run_comparison(num_episodes=50)

    # Print results
    print("\n=== FINAL EVALUATION RESULTS ===")
    for agent_name, metrics in results.items():
        print(f"\n{agent_name}:")
        for metric, value in metrics.items():
            print(f"  {metric}: {value:.4f}")

    # Plot results
    evaluator.plot_results(results)

    print("Training completed!")
    print(f"Logs saved in: {logger.log_dir}")
    print("Checkpoints saved: wifi_rl_checkpoint.pth, wifi_rl_best_model.pth")

    return agent, results

In [ ]:
def load_trained_model(model_path: str, env, logger):
    """Load a trained model"""
    checkpoint = torch.load(model_path, weights_only=False)

    # Recreate agent
    agent = MAPPOAgent(env, checkpoint['training_config'], logger)
    agent.policy_net.load_state_dict(checkpoint['model_state_dict'])

    if 'optimizer_state_dict' in checkpoint:
        agent.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    if 'training_metrics' in checkpoint:
        agent.training_metrics = checkpoint['training_metrics']
    if 'update_count' in checkpoint:
        agent.update_count = checkpoint['update_count']
    if 'best_reward' in checkpoint:
        agent.best_reward = checkpoint['best_reward']

    return agent

# TRAINING

In [ ]:
def run_full_training():
    """Full-scale production training of WiFi RL agents"""
    print("==== WiFi 802.11 RL Production Training ====")

    # Initialize CSV logger
    logger = CSVLogger()

    # Production training configuration
    config = {
        'hidden_dim': 256,           # Larger network
        'learning_rate': 2e-4,       # Lower learning rate for stability
        'gamma': 0.995,              # Higher discount for long-term planning
        'gae_lambda': 0.95,
        'clip_ratio': 0.15,          # Slightly tighter clipping
        'entropy_coef': 0.005,       # Lower entropy for exploitation
        'value_coef': 0.5,
        'max_grad_norm': 0.5
    }

    # Multi-scenario curriculum training
    scenarios = [
        # Easy: Small office
        {
            'num_aps': 3,
            'stations_per_ap': 4,
            'area_size': (50, 50),
            'topology': 'grid',
            'max_steps': 800,
            'name': 'Small Office'
        },
        # Medium: Standard office
        {
            'num_aps': 6,
            'stations_per_ap': 8,
            'area_size': (80, 80),
            'topology': 'grid',
            'max_steps': 1000,
            'name': 'Standard Office'
        },
        # Hard: Dense apartment
        {
            'num_aps': 8,
            'stations_per_ap': 12,
            'area_size': (60, 60),
            'topology': 'grid',
            'max_steps': 1200,
            'name': 'Dense Apartment'
        },
        # Very Hard: High-density hotspot
        {
            'num_aps': 12,
            'stations_per_ap': 15,
            'area_size': (40, 40),
            'topology': 'grid',
            'max_steps': 1500,
            'name': 'High-Density Hotspot'
        }
    ]

    print(f"Curriculum training with {len(scenarios)} scenarios")
    for i, scenario in enumerate(scenarios):
        print(f"  Scenario {i+1}: {scenario['name']} - {scenario['num_aps']} APs, "
              f"{scenario['num_aps'] * scenario['stations_per_ap']} stations")

    # Training parameters
    total_timesteps = 2000000  # 2M timesteps total
    timesteps_per_scenario = total_timesteps // len(scenarios)
    steps_per_update = 4096    # Larger batches for stability

    print(f"\nTotal training: {total_timesteps:,} timesteps")
    print(f"Per scenario: {timesteps_per_scenario:,} timesteps")
    print(f"Batch size: {steps_per_update} steps")

    # Initialize with first scenario
    env = WiFiEnvironment(scenarios[0])
    agent = MAPPOAgent(env, config, logger)

    print(f"\nUsing device: {agent.device}")
    print(f"Network architecture: {sum(p.numel() for p in agent.policy_net.parameters())} parameters")

    # Curriculum training loop
    for scenario_idx, scenario in enumerate(scenarios):
        print(f"\n{'='*60}")
        print(f"SCENARIO {scenario_idx + 1}/{len(scenarios)}: {scenario['name']}")
        print(f"{'='*60}")

        # Create environment for this scenario
        env = WiFiEnvironment(scenario)
        agent.env = env

        # Adaptive learning rate decay
        if scenario_idx > 0:
            for param_group in agent.optimizer.param_groups:
                param_group['lr'] *= 0.8  # Reduce learning rate by 20%
            print(f"Reduced learning rate to: {param_group['lr']:.2e}")

        # Train on this scenario
        scenario_start_updates = agent.update_count
        agent.train(timesteps_per_scenario, steps_per_update)

        # Scenario evaluation
        print(f"\nEvaluating scenario {scenario_idx + 1}...")
        avg_reward = agent.evaluate(num_episodes=10)

        updates_this_scenario = agent.update_count - scenario_start_updates
        print(f"Scenario {scenario_idx + 1} completed:")
        print(f"  Updates: {updates_this_scenario}")
        print(f"  Final reward: {avg_reward:.3f}")
        print(f"  Best reward so far: {agent.best_reward:.3f}")

    # Final comprehensive evaluation
    print(f"\n{'='*60}")
    print("FINAL EVALUATION")
    print(f"{'='*60}")

    # Test on all scenarios
    final_results = {}

    for scenario_idx, scenario in enumerate(scenarios):
        print(f"\nTesting on {scenario['name']}...")
        test_env = WiFiEnvironment(scenario)
        agent.env = test_env

        evaluator = EvaluationSuite(test_env, agent, logger)
        scenario_results = evaluator.run_comparison(num_episodes=25)
        final_results[scenario['name']] = scenario_results

        # Print scenario results
        print(f"\nResults for {scenario['name']}:")
        for agent_name, metrics in scenario_results.items():
            print(f"  {agent_name}:")
            print(f"    Reward: {metrics['mean_reward']:.4f}")
            print(f"    Throughput: {metrics['mean_throughput']:.2f} Mbps")
            print(f"    Latency: {metrics['mean_latency']:.2f} ms")
            print(f"    Fairness: {metrics['mean_fairness']:.3f}")

    # Overall performance summary
    print(f"\n{'='*60}")
    print("TRAINING SUMMARY")
    print(f"{'='*60}")

    # Calculate average performance across all scenarios
    avg_metrics = {}
    for agent_name in ['MAPPO', 'Random', 'Least Congested']:
        if agent_name in final_results[scenarios[0]['name']]:
            avg_metrics[agent_name] = {
                'mean_reward': np.mean([final_results[s['name']][agent_name]['mean_reward']
                                      for s in scenarios]),
                'mean_throughput': np.mean([final_results[s['name']][agent_name]['mean_throughput']
                                          for s in scenarios]),
                'mean_latency': np.mean([final_results[s['name']][agent_name]['mean_latency']
                                       for s in scenarios]),
                'mean_fairness': np.mean([final_results[s['name']][agent_name]['mean_fairness']
                                        for s in scenarios])
            }

    print(f"Average performance across all scenarios:")
    for agent_name, metrics in avg_metrics.items():
        print(f"\n{agent_name}:")
        print(f"  Reward: {metrics['mean_reward']:.4f}")
        print(f"  Throughput: {metrics['mean_throughput']:.2f} Mbps")
        print(f"  Latency: {metrics['mean_latency']:.2f} ms")
        print(f"  Fairness: {metrics['mean_fairness']:.3f}")

    # Performance improvements
    if 'MAPPO' in avg_metrics and 'Random' in avg_metrics:
        reward_improvement = ((avg_metrics['MAPPO']['mean_reward'] -
                             avg_metrics['Random']['mean_reward']) /
                            abs(avg_metrics['Random']['mean_reward'])) * 100
        throughput_improvement = ((avg_metrics['MAPPO']['mean_throughput'] -
                                 avg_metrics['Random']['mean_throughput']) /
                                avg_metrics['Random']['mean_throughput']) * 100

        print(f"\nMAPPO vs Random baseline:")
        print(f"  Reward improvement: {reward_improvement:.1f}%")
        print(f"  Throughput improvement: {throughput_improvement:.1f}%")

    # Create comprehensive results plot
    evaluator = EvaluationSuite(env, agent, logger)
    evaluator.plot_results(avg_metrics, 'wifi_rl_final_results.png')

    # Training completion summary
    print(f"\n{'='*60}")
    print("TRAINING COMPLETED")
    print(f"{'='*60}")
    print(f"Total updates: {agent.update_count}")
    print(f"Best reward achieved: {agent.best_reward:.3f}")
    print(f"Logs directory: {logger.log_dir}")
    print(f"Final model: wifi_rl_best_model.pth")
    print(f"Latest checkpoint: wifi_rl_checkpoint.pth")
    print(f"Results plot: wifi_rl_final_results.png")

    return agent, final_results

In [ ]:
trained_agent, evaluation_results = run_full_training()

==== WiFi 802.11 RL Production Training ====
Curriculum training with 4 scenarios
  Scenario 1: Small Office - 3 APs, 12 stations
  Scenario 2: Standard Office - 6 APs, 48 stations
  Scenario 3: Dense Apartment - 8 APs, 96 stations
  Scenario 4: High-Density Hotspot - 12 APs, 180 stations

Total training: 2,000,000 timesteps
Per scenario: 500,000 timesteps
Batch size: 4096 steps
Initialized network: 3 APs, 12 STAs

Using device: cpu
Network architecture: 976660 parameters

SCENARIO 1/4: Small Office
Initialized network: 3 APs, 12 STAs
Starting MAPPO training: 500000 timesteps
Episode 0: reward=467.069, throughput=2.74 Mbps, latency=8.5 ms, steps=800
Episode 1: reward=470.360, throughput=3.23 Mbps, latency=9.2 ms, steps=800
Episode 2: reward=475.302, throughput=2.92 Mbps, latency=10.8 ms, steps=800
Episode 3: reward=470.400, throughput=3.31 Mbps, latency=13.4 ms, steps=800
Episode 4: reward=469.310, throughput=2.45 Mbps, latency=10.5 ms, steps=800
Update 1: policy_loss=0.2769, value_los

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = "/content/drive/MyDrive/WiFi_Generalization/wifi_rl_logs"
os.makedirs(BASE_DIR, exist_ok=True)

print("Generalization folder:", BASE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Generalization folder: /content/drive/MyDrive/WiFi_Generalization/wifi_rl_logs


In [ ]:
MODEL_PATH = os.path.join(
    BASE_DIR,
    "wifi_rl_best_model.pth"
)

print("Model path:", MODEL_PATH)
if os.path.exists(MODEL_PATH):
    print("✓ Trained model found")
    print(f"Size: {os.path.getsize(MODEL_PATH) / (1024**2):.2f} MB")
else:
    print("✗ Model not found")
    print("Expected:", MODEL_PATH)
GENERALIZATION_DIR = os.path.join(
    BASE_DIR,
    "generalization_results"
)

os.makedirs(GENERALIZATION_DIR, exist_ok=True)

print("Results will be saved to:")
print(GENERALIZATION_DIR)

Model path: /content/drive/MyDrive/WiFi_Generalization/wifi_rl_logs/wifi_rl_best_model.pth
✓ Trained model found
Size: 3.74 MB
Results will be saved to:
/content/drive/MyDrive/WiFi_Generalization/wifi_rl_logs/generalization_results


In [ ]:
# Cell 32: Create base environment for combined test

combined_config = {
    'num_aps': 20,
    'stations_per_ap': 15,
    'area_size': (50, 50),
    'topology': 'grid',
    'max_steps': 1200,
    'name': 'Combined Generalization'
}

combined_env = WiFiEnvironment(combined_config)

print("=" * 60)
print("COMBINED GENERALIZATION - BASE ENVIRONMENT")
print("=" * 60)

print(f"APs: {combined_env.num_aps}")
print(f"STAs: {len(combined_env.stations)}")
print(f"Max steps: {combined_env.max_steps}")

Initialized network: 20 APs, 300 STAs
COMBINED GENERALIZATION - BASE ENVIRONMENT
APs: 20
STAs: 300
Max steps: 1200


In [ ]:
# Cell 33: Create unseen random AP layout safely

random.seed(2026)
np.random.seed(2026)

for ap in combined_env.aps.values():
    ap.position = (
        random.uniform(5, combined_env.area_size[0] - 5),
        random.uniform(5, combined_env.area_size[1] - 5)
    )

# Move each STA around its associated AP
for sta in combined_env.stations.values():
    ap_pos = combined_env.aps[sta.ap_id].position

    angle = random.uniform(0, 2 * math.pi)
    distance = random.uniform(5, 20)

    sta.position = (
        ap_pos[0] + distance * math.cos(angle),
        ap_pos[1] + distance * math.sin(angle)
    )

print("✓ AP positions replaced with unseen random layout")
print("✓ STA positions updated around their associated APs")

✓ AP positions replaced with unseen random layout
✓ STA positions updated around their associated APs


In [ ]:
# Cell 34: Verify combined network layout

print("=" * 60)
print("COMBINED NETWORK VERIFICATION")
print("=" * 60)

assert combined_env.num_aps == 20
assert len(combined_env.stations) == 300

print(f"✓ APs: {combined_env.num_aps}")
print(f"✓ STAs: {len(combined_env.stations)}")

print("\nSample AP positions:")
for i in range(min(5, combined_env.num_aps)):
    print(f"AP {i+1}: {combined_env.aps[i].position}")

COMBINED NETWORK VERIFICATION
✓ APs: 20
✓ STAs: 300

Sample AP positions:
AP 1: (9.764795398558523, 25.100630209250024)
AP 2: (25.472908510922842, 39.40002350597102)
AP 3: (9.105474020278393, 13.931382267119625)
AP 4: (29.041226095403463, 27.262360876981898)
AP 5: (36.33493529932761, 26.912458194017926)


In [ ]:
# Cell 35: Increase interference for combined scenario

for source in combined_env.interference_sources:
    source.power = min(source.power + 10.0, 20.0)
    source.duty_cycle = min(source.duty_cycle * 1.5, 1.0)

print("✓ High-interference condition applied")
print(f"✓ Number of interference sources: "
      f"{len(combined_env.interference_sources)}")

for i, source in enumerate(combined_env.interference_sources):
    print(
        f"Source {i+1}: "
        f"power={source.power:.2f} dBm, "
        f"duty_cycle={source.duty_cycle:.2f}"
    )

✓ High-interference condition applied
✓ Number of interference sources: 1
Source 1: power=17.59 dBm, duty_cycle=0.16


In [ ]:
# Cell 36: Combined non-stationary traffic scheduler

COMBINED_TRAFFIC_SCHEDULE = [
    (0,    300, 0.50),
    (300,  600, 2.00),
    (600,  900, 0.50),
    (900, 1200, 2.00)
]

combined_env._combined_traffic_original = combined_env._generate_traffic


def combined_generate_traffic(self):

    multiplier = 1.0

    for start, end, factor in COMBINED_TRAFFIC_SCHEDULE:
        if start <= self.current_step < end:
            multiplier = factor
            break

    for sta in self.stations.values():

        base_rate = Config.ARRIVAL_RATES[sta.traffic_type]
        arrival_rate = base_rate * multiplier
        arrival_prob = arrival_rate * self.time_step_duration

        packets = np.random.poisson(arrival_prob)
        sta.queue_length += packets


combined_env._generate_traffic = combined_generate_traffic.__get__(
    combined_env,
    WiFiEnvironment
)

print("✓ Non-stationary traffic applied")
print("✓ Low → High → Low → High")

✓ Non-stationary traffic applied
✓ Low → High → Low → High


In [ ]:
# Cell 37: Verify combined generalization setup

print("=" * 60)
print("COMBINED GENERALIZATION VERIFICATION")
print("=" * 60)

print(f"APs: {combined_env.num_aps}")
print(f"STAs: {len(combined_env.stations)}")
print(f"Area: {combined_env.area_size}")

print("\nConditions:")
print("✓ Larger network: 20 APs / 300 STAs")
print("✓ Unseen topology: random AP deployment")
print("✓ Non-stationary traffic: LOW/HIGH alternating")
print("✓ High interference: increased power/duty cycle")
print("✓ Retraining: NO")

print("\nTraffic schedule:")
for start, end, factor in COMBINED_TRAFFIC_SCHEDULE:
    level = "LOW" if factor < 1 else "HIGH"
    print(f"  {start:4d}-{end:4d}: {level} (x{factor})")

COMBINED GENERALIZATION VERIFICATION
APs: 20
STAs: 300
Area: (50, 50)

Conditions:
✓ Larger network: 20 APs / 300 STAs
✓ Unseen topology: random AP deployment
✓ Non-stationary traffic: LOW/HIGH alternating
✓ High interference: increased power/duty cycle
✓ Retraining: NO

Traffic schedule:
     0- 300: LOW (x0.5)
   300- 600: HIGH (x2.0)
   600- 900: LOW (x0.5)
   900-1200: HIGH (x2.0)


In [ ]:
# Cell 38: Load trained model for combined evaluation

combined_logger = CSVLogger(
    os.path.join(
        GENERALIZATION_DIR,
        "combined_generalization_logs"
    )
)

combined_agent = load_trained_model(
    MODEL_PATH,
    combined_env,
    combined_logger
)

combined_agent.policy_net.eval()

print("\n✓ Trained model loaded")
print("✓ Evaluation only")
print("✓ No retraining")


✓ Trained model loaded
✓ Evaluation only
✓ No retraining


In [ ]:
# Cell 39: Combined generalization quick test

print("=" * 70)
print("COMBINED GENERALIZATION - QUICK TEST")
print("=" * 70)

print("Conditions:")
print("  • Unseen random topology")
print("  • Larger network: 20 APs / 300 STAs")
print("  • Non-stationary traffic")
print("  • High interference")
print("  • No retraining")
print("=" * 70)

combined_quick_reward = combined_agent.evaluate(
    num_episodes=50
)

print("\n✓ Combined quick evaluation completed")
print(f"Average reward: {combined_quick_reward:.3f}")

COMBINED GENERALIZATION - QUICK TEST
Conditions:
  • Unseen random topology
  • Larger network: 20 APs / 300 STAs
  • Non-stationary traffic
  • High interference
  • No retraining
Evaluation Results:
  Average Reward: 20365.470
  Average Throughput: 131.75 Mbps
  Average Latency: 20.13 ms
  Average Collision Rate: 0.052
  Average Fairness: 0.432

✓ Combined quick evaluation completed
Average reward: 20365.470
